In [12]:
import cplex
import pandas as pd

# Cargar los datos
partidos = pd.read_csv('partidos.csv')  # Contiene info sobre partidos locales y visitantes
puntajes_iniciales = pd.read_csv('puntajes_iniciales.csv')  # Puntajes iniciales de los países
combinaciones = pd.read_csv('combinaciones.csv')  # Combinaciones de países

# Obtener la lista de equipos y partidos
equipos = puntajes_iniciales['id'].unique()
partidos_ids = partidos.index

# Inicializar el modelo de CPLEX
modelo = cplex.Cplex()
argentina_idx = 1

# Definir las variables de decisión
home_vars = [f"home_{i}" for i in partidos_ids]
away_vars = [f"away_{i}" for i in partidos_ids]
draw_vars = [f"draw_{i}" for i in partidos_ids]
puntajes_vars = [f"puntaje_{c}" for c in equipos]
delta_vars = [f"delta_{k}" for k in combinaciones['id']]

# Añadir las variables al modelo
modelo.variables.add(names=home_vars, lb=[0] * len(home_vars), ub=[1] * len(home_vars), types=['B'] * len(home_vars))
modelo.variables.add(names=away_vars, lb=[0] * len(away_vars), ub=[1] * len(away_vars), types=['B'] * len(away_vars))
modelo.variables.add(names=draw_vars, lb=[0] * len(draw_vars), ub=[1] * len(draw_vars), types=['B'] * len(draw_vars))
modelo.variables.add(names=puntajes_vars, lb=[0] * len(puntajes_vars), ub=[1000] * len(puntajes_vars), types=['C'] * len(puntajes_vars))
modelo.variables.add(names=delta_vars, lb=[0] * len(delta_vars), ub=[1] * len(delta_vars), types=['B'] * len(delta_vars))

# Función objetivo: maximizar delta_1
modelo.objective.set_sense(modelo.objective.sense.maximize)
terms = []

for k in combinaciones['id']:
    terms.append((f"delta_{k}", 1))

modelo.objective.set_linear([terms])

# Restricciones
# 1. away_i = 1 para todos los partidos locales de Argentina
locales_argentina = partidos[(partidos['home_id'] == argentina_idx)].index
for i in locales_argentina:
    modelo.linear_constraints.add(
        lin_expr=[cplex.SparsePair(ind=[f"away_{i}"], val=[1])],
        senses=['E'],  # Igualdad
        rhs=[1]
    )

# 2. home_i = 1 para todos los partidos donde Argentina es visitante
visitantes_argentina = partidos[(partidos['away_id'] == argentina_idx)].index
for i in visitantes_argentina:
    modelo.linear_constraints.add(
        lin_expr=[cplex.SparsePair(ind=[f"home_{i}"], val=[1])],
        senses=['E'],
        rhs=[1]
    )

# 3. home_i + away_i + draw_i = 1 para todos los partidos
for i in partidos_ids:
    modelo.linear_constraints.add(
        lin_expr=[cplex.SparsePair(ind=[f"home_{i}", f"away_{i}", f"draw_{i}"], val=[1, 1, 1])],
        senses=['E'],
        rhs=[1]
    )

# 4. Calcular puntaje de cada país
# puntajes = {}
for c in equipos:
    ind = [f"puntaje_{c}"]
    val = [1]

    # Calcular puntaje de cada país
    partidos_locales = partidos[(partidos['home_id'] == c)].index

    for i in partidos_locales:
       ind.append(f"home_{i}")
       val.append(-3)
    
    partidos_visitantes = partidos[(partidos['away_id'] == c)].index
    for i in partidos_visitantes:
        ind.append(f"away_{i}")
        val.append(-3)

    for i in partidos_locales.union(partidos_visitantes):
        ind.append(f"draw_{i}")
        val.append(-1)

    modelo.linear_constraints.add(
        lin_expr=[cplex.SparsePair(ind=ind, val=val)],
        senses=['E'],
        rhs=[float(puntajes_iniciales[puntajes_iniciales['id'] == c]['puntaje'].values[0])]
    )

# 5. Restricción: puntaje de Argentina debe ser menor al puntaje de otros países en cada combinación
M = 1000  # Constante grande para el modelo
for _, comb in combinaciones.iterrows():
    k = comb['id']
    paises_k = comb[1:].dropna().values
    for c in paises_k:
        modelo.linear_constraints.add(
            lin_expr=[cplex.SparsePair(
                ind=[f"puntaje_{argentina_idx}", f"puntaje_{c}", f"delta_{k}"],
                val=[1, -1, M]
            )],
            senses=['L'],
            rhs=[M]
        )




In [14]:
# Resolver el modelo
modelo.solve()

# Resultados
sol = modelo.solution
print(f"Solución óptima: {sol.get_objective_value()}")
for i in partidos_ids:
    print(f"Partido {i}: home={sol.get_values(f'home_{i}')}, away={sol.get_values(f'away_{i}')}, draw={sol.get_values(f'draw_{i}')}")
for k in combinaciones['id']:
    print(f"Combinación {k}: delta={sol.get_values(f'delta_{k}')}")

values_deltas = [sol.get_values(f'delta_{k}') for k in combinaciones['id']]

if all([v == 1 for v in values_deltas]):
    print("Argentina todavía no tiene garantizado el pase al mundial")

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Tried aggregator 2 times.
MIP Presolve eliminated 546 rows and 181 columns.
MIP Presolve modified 22 coefficients.
Aggregator did 5 substitutions.
Reduced MIP has 11 rows, 28 columns, and 43 nonzeros.
Reduced MIP has 28 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.53 ticks)
Found incumbent of value 1.000000 after 0.03 sec. (0.63 ticks)

Root node processing (before b&c):
  Real time             =    0.03 sec. (0.63 ticks)
Parallel b&c, 16 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.03 sec. (0.63 ticks)
Solución óptima: 1.0
Partido 0: home=0.0, away=1.0, draw=0.0
Partido 1: home=1.0, away=0.0, draw=0.0
Partido 2: home=1.0, away=0.0, draw=0.0
Partido 3: home=1.0, away=0.0, draw=0.0
Partido 4